# Operations Research Group Project

## Scope

**Why Dual Simplex**:

The baseline LP from `00_Linear_Programming.ipynb` produces a theoretically optimal but unrealistic solution — it exploits fluid milk and cooking oils in unrealistically large quantities. When food group acceptability constraints are added:

$$\sum_{i \in I_g} x_i \leq U_g \quad \forall g \in G$$

the baseline optimal solution $x^*$ violates these new constraints. This creates the exact initialisation condition for the Dual Simplex Method:

- **Dual feasibility preserved** — Row 0 reduced costs are unaffected by adding $\leq$ constraints, so the basis remains dual feasible
- **Primal feasibility broken** — $x^*$ violates the new group caps, so the basis is primal infeasible

From the lecture: *While dealing with standard simplex, if we face any breaks, it is useful to check dual feasible.*

**Dual Simplex Initialisation Condition**:

$$\text{Dual feasible} \;\cap\; \text{Primal infeasible} \implies \text{Apply Dual Simplex}$$

**Dual Simplex Algorithm (from lecture)**:

1. Initialise — verify current basis is dual feasible (all Row 0 coefficients $\geq 0$)
2. Feasibility test — identify the most negative basic variable (leaving variable)
3. Ratio test — select entering variable to maintain dual feasibility (Row 0 stays $\geq 0$)
4. Pivot — restore primal feasibility
5. Repeat until all basic variables $\geq 0$

**Primal Infeasibility Check**:

For each food group $g$, the violation of the baseline solution $x^*$ is:

$$\text{violation}_g = \sum_{i \in I_g} x_i^* - U_g$$

A positive violation means the group cap constraint is broken and the baseline basis is primal infeasible with respect to that constraint.

**Cost of Acceptability**:

$$\Delta Z = Z^*_{\text{capped}} - Z^*_{\text{baseline}} > 0$$

This is the additional cost imposed by requiring a realistic meal structure. It quantifies the trade-off between cost optimality and meal acceptability.

**Index Sets**:

$i \in I = \{1, 2, \ldots, 204\}$ — food items

$j \in J = \{1, 2, \ldots, 29\}$ — nutrients

$g \in G$ — food groups $\{$fluid\_milk, cooking\_oils, grain\_products, leafy\_veg, protein\_foods, fruits$\}$

$U_g$ — gram cap for group $g$

In [1]:
# Necessary imports
import numpy as np
import pandas as pd

from scipy.optimize import linprog, OptimizeResult
from tabulate import tabulate

import os
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Data loc
PATH_PRICES    = "/Users/julianyoo/Downloads/Study/PWR/Term 1/Operations Research/Project/data/food_prices_200.xlsx"
PATH_NUTRITION = "/Users/julianyoo/Downloads/Study/PWR/Term 1/Operations Research/Project/data/food_nutrition_200.xlsx"
PATH_REQ_LB    = "/Users/julianyoo/Downloads/Study/PWR/Term 1/Operations Research/Project/data/dri_lunch_lb.csv"
PATH_REQ_UB    = "/Users/julianyoo/Downloads/Study/PWR/Term 1/Operations Research/Project/data/dri_lunch_ub.csv"

# Portion cap
MAX_GRAMS_PER_FOOD = 200.0

In [3]:
# Data load
def load_data() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Loading all required data

    Tap water should be excluded

    Returns:
    nutrition   : (204, 35) - food nutritional content per 100g
    prices      : (204, 13) - food prices per gram
    lb          : (5, 29+4) - lower bounds per age-sex group
    ub          : (5, 29+4) - upper bounds per age-sex group
    """
    nutrition = pd.read_excel(PATH_NUTRITION, keep_default_na=False)
    prices    = pd.read_excel(PATH_PRICES,    keep_default_na=False)
    lb        = pd.read_csv(PATH_REQ_LB).set_index("age_sex_group")
    ub        = pd.read_csv(PATH_REQ_UB).set_index("age_sex_group")

    # Remove water
    nutrition = nutrition[nutrition["food_name"] != "Water, Tap"].reset_index(drop=True)
    prices    = prices[prices["food_name"] != "Water, Tap"].reset_index(drop=True)

    return nutrition, prices, lb, ub

In [4]:
nutrition, prices, lb, ub = load_data()
print(f"Foods     : {len(nutrition)}")
print(f"Nutrients : {len(nutrition.columns) - 6}")
print(f"Groups    : {list(lb.index)}")

Foods     : 204
Nutrients : 29
Groups    : ['children_4_8_MF', 'males_9_13', 'females_9_13', 'males_14_18', 'females_14_18']


In [5]:
# Build LP coefficient matrices
def build_matrices(
        nutrition: pd.DataFrame,
        prices:    pd.DataFrame
) -> tuple[np.ndarray, np.ndarray, list[str], list[str]]:
    """
    Constructing the obj vec c and const coeff matrix A

    Notation:
    n   : number of foods
    m   : number of nutrients
    c   : cost per gram (USD/g)
    A   : nutrient per gram

    Returns:
    c           : (n,) - obj coeff (USD per gram)
    A           : (n, m) - nutrient per gram matrix
    food_names  : list[str]
    nut_cols    : list[str]
    """
    nut_cols = list(nutrition.columns[6:])  # 29 nutrients

    merged = nutrition.merge(
        prices[["food_id", "price_per_1g_usd"]],
        on="food_id",
        how="inner"
    )

    food_names = list(merged["food_name"])
    c          = merged["price_per_1g_usd"].to_numpy(dtype=float)
    A          = merged[nut_cols].to_numpy(dtype=float) / 100.0

    return c, A, food_names, nut_cols

In [6]:
c, A, food_names, nut_cols = build_matrices(nutrition, prices)
print(f"Obj vector c : {c.shape}     (USD/g)")
print(f"Const mat A  : {A.shape}     (nutrient/g)")
print(f"Cost range   : ${c.min():.4f} - ${c.max():.4f} per gram")

Obj vector c : (204,)     (USD/g)
Const mat A  : (204, 29)     (nutrient/g)
Cost range   : $0.0001 - $0.0500 per gram


In [7]:
# Build nutrient constraints
def get_constraints() -> dict[str, list[str]]:
    """
    Classifying each nutrient into LP constraint types

    Categories:
    min_only    : sum_i a_ij x_i >= LB_j   (RDA/AI minimum requirements)
    max_only    : sum_i a_ij x_i <= UB_j   (NSLP/DGA ceilings)
    range       : LB_j <= sum_i a_ij x_i <= UB_j  (energy + fat)
    skip        : no numeric DRI -- excluded from LP constraints
    """
    min_only = [
        "protein_g",
        "carbohydrates_g",
        "dietary_fiber_g",
        "calcium_mg",
        "iron_mg",
        "magnesium_mg",
        "phosphorus_mg",
        "potassium_mg",
        "zinc_mg",
        "vitamin_a_ug_rae",
        "vitamin_d_ug",
        "vitamin_e_mg_ate",
        "vitamin_k_ug",
        "vitamin_c_mg",
        "thiamin_b1_mg",
        "riboflavin_b2_mg",
        "niacin_b3_mg",
        "vitamin_b6_mg",
        "folate_ug_dfe",
        "vitamin_b12_ug",
        "selenium_ug",
    ]
    max_only = [
        "sugars_g",
        "saturated_fat_g",
        "sodium_mg",
    ]
    range_nuts = [
        "energy_kcal",
        "total_fat_g",
    ]
    skip = [
        "monounsaturated_fat_g",
        "polyunsaturated_fat_g",
        "cholesterol_mg",
    ]

    return {
        "min_only" : min_only,
        "max_only" : max_only,
        "range"    : range_nuts,
        "skip"     : skip,
    }

In [8]:
constraint_sets = get_constraints()
for ctype, nuts in constraint_sets.items():
    print(f"{ctype}: {len(nuts):2d} nutrients")

min_only: 21 nutrients
max_only:  3 nutrients
range:  2 nutrients
skip:  3 nutrients


In [9]:
# Food group caps
def get_group_caps(food_names: list[str]) -> dict[str, tuple[list[int], float]]:
    """
    Define food group caps for meal realism.

    For each group: a list of food indices and a combined gram cap.
    The LP must spread across food groups — no single category can dominate.

    Groups and caps:
    fluid_milk      — one standard school carton (all milk types combined)
    cooking_oils    — incidental cooking use only
    grain_products  — one grain serving (all grains/cereals combined)
    leafy_veg       — one side salad portion
    protein_foods   — one protein serving
    fruits          — one fruit serving

    Returns:
    dict mapping group_name -> (list of food indices in group, gram cap)
    """
    group_keywords = {
        "fluid_milk"    : ["milk, 2%", "milk, whole", "milk, 1%", "milk, nonfat",
                           "milk, skim", "milk, soy", "milk, almond", "milk, chocolate",
                           "milk,"],
        "cooking_oils"  : ["vegetable oil", "olive oil", "canola oil", "coconut oil"],
        "grain_products": ["cornmeal", "corn flakes", "raisin bran", "oatmeal",
                           "granola", "rice, white", "rice, brown", "pasta",
                           "spaghetti", "macaroni", "bread,", "tortilla",
                           "bagel", "pita", "quinoa", "barley", "millet",
                           "couscous", "buckwheat", "crackers", "wild rice"],
        "leafy_veg"     : ["spinach", "kale", "romaine", "lettuce", "collard",
                           "swiss chard", "bok choy", "mixed greens"],
        "protein_foods" : ["chicken", "beef", "pork", "turkey", "tuna",
                           "salmon", "tilapia", "cod", "shrimp", "sardine",
                           "mackerel", "clam", "crab", "catfish", "liver",
                           "egg", "tofu", "edamame"],
        "fruits"        : ["banana", "apple", "orange", "grape", "strawberr",
                           "blueberr", "peach", "pear", "pineapple", "watermelon",
                           "cantaloupe", "mango", "kiwi", "cherry", "plum",
                           "grapefruit", "pomegranate", "raspberry"],
    }

    group_caps = {
        "fluid_milk"    : 150.0,
        "cooking_oils"  : 3.0,
        "grain_products": 120.0,
        "leafy_veg"     : 80.0,
        "protein_foods" : 100.0,
        "fruits"        : 120.0,
    }

    groups = {}
    food_names_lower = [f.lower() for f in food_names]

    for group, keywords in group_keywords.items():
        indices = [
            i for i, name in enumerate(food_names_lower)
            if any(kw in name for kw in keywords)
        ]
        groups[group] = (indices, group_caps[group])

    return groups

In [10]:
groups = get_group_caps(food_names)

for group, (indices, cap) in groups.items():
    print(f"{group:20s} : cap = {cap:5.0f} g  |  {len(indices):3d} foods")

fluid_milk           : cap =   150 g  |    7 foods
cooking_oils         : cap =     3 g  |    4 foods
grain_products       : cap =   120 g  |   27 foods
leafy_veg            : cap =    80 g  |    9 foods
protein_foods        : cap =   100 g  |   47 foods
fruits               : cap =   120 g  |   21 foods


In [11]:
# Build A_ub — nutrient constraints only (baseline, no group caps)
def build_nutrient_system(
        A:               np.ndarray,
        nut_cols:        list[str],
        lb_row:          pd.Series,
        ub_row:          pd.Series,
        constraint_sets: dict[str, list[str]],
) -> tuple[np.ndarray, np.ndarray, list[tuple]]:
    """
    Assembles only the nutrient constraint rows — no group caps.
    This is the BASELINE system from 00_Linear_Programming.

    Encoding:
    min constraint  (Ax >= b)  -->  -Ax <= -b
    max constraint  (Ax <= b)  -->   Ax <=  b
    range lb        (Ax >= b)  -->  -Ax <= -b
    range ub        (Ax <= b)  -->   Ax <=  b

    Returns:
    A_ub    : (28, n) nutrient constraint matrix
    b_ub    : (28,)   nutrient constraint RHS
    names   : list of (constraint_type, nutrient, rhs_value)
    """
    nut_idx = {nut: j for j, nut in enumerate(nut_cols)}

    A_ub_rows, b_ub_rows, names = [], [], []

    def _valid(val) -> bool:
        if val == "": return False
        try: return not np.isnan(float(val))
        except: return False

    for nut in constraint_sets["min_only"]:
        j   = nut_idx[nut]
        val = lb_row.get(nut, "")
        if not _valid(val): continue
        A_ub_rows.append(-A[:, j]); b_ub_rows.append(-float(val))
        names.append(("min_lb", nut, float(val)))

    for nut in constraint_sets["max_only"]:
        j   = nut_idx[nut]
        val = ub_row.get(nut, "")
        if not _valid(val): continue
        A_ub_rows.append(A[:, j]); b_ub_rows.append(float(val))
        names.append(("max_ub", nut, float(val)))

    for nut in constraint_sets["range"]:
        j     = nut_idx[nut]
        v_lb  = lb_row.get(nut, "")
        v_ub  = ub_row.get(nut, "")
        if _valid(v_lb):
            A_ub_rows.append(-A[:, j]); b_ub_rows.append(-float(v_lb))
            names.append(("range_lb", nut, float(v_lb)))
        if _valid(v_ub):
            A_ub_rows.append(A[:, j]); b_ub_rows.append(float(v_ub))
            names.append(("range_ub", nut, float(v_ub)))

    return np.array(A_ub_rows), np.array(b_ub_rows), names

In [12]:
# Solve BASELINE LP (no group caps) — primal simplex
def solve_baseline(
        group_id:        str,
        c:               np.ndarray,
        A:               np.ndarray,
        food_names:      list[str],
        nut_cols:        list[str],
        lb_row:          pd.Series,
        ub_row:          pd.Series,
        constraint_sets: dict[str, list[str]],
) -> dict:
    """
    Solve the baseline LP without group caps.

    This is the starting basis for the dual simplex analysis.
    The resulting solution x* is dual feasible (optimal Row 0)
    but will be primal infeasible once group caps are introduced.

    Returns:
    dict with group_id, status, cost_usd, primal_x, nit, A_ub, b_ub, food_names
    """
    n = len(food_names)

    A_ub, b_ub, names = build_nutrient_system(
        A, nut_cols, lb_row, ub_row, constraint_sets
    )
    bounds = [(0.0, MAX_GRAMS_PER_FOOD)] * n

    res: OptimizeResult = linprog(
        c      = c,
        A_ub   = A_ub,
        b_ub   = b_ub,
        bounds = bounds,
        method = "highs",
    )

    if res.status != 0:
        return {"group_id": group_id, "status": res.message}

    return {
        "group_id"   : group_id,
        "status"     : "Optimal",
        "cost_usd"   : round(float(res.fun), 6),
        "primal_x"   : res.x,
        "nit"        : res.nit,
        "A_ub"       : A_ub,
        "b_ub"       : b_ub,
        "con_names"  : names,
        "food_names" : food_names,
    }

In [13]:
# Solve baseline for all groups
baseline_results = []

for group_id in lb.index:
    result = solve_baseline(
        group_id        = group_id,
        c               = c,
        A               = A,
        food_names      = food_names,
        nut_cols        = nut_cols,
        lb_row          = lb.loc[group_id],
        ub_row          = ub.loc[group_id],
        constraint_sets = constraint_sets,
    )
    baseline_results.append(result)
    print(f"{group_id:25s}  Z*_baseline = ${result['cost_usd']:.6f}  nit = {result['nit']}")

children_4_8_MF            Z*_baseline = $0.687245  nit = 19
males_9_13                 Z*_baseline = $0.796206  nit = 19
females_9_13               Z*_baseline = $0.773432  nit = 14
males_14_18                Z*_baseline = $0.927501  nit = 19
females_14_18              Z*_baseline = $0.875591  nit = 18


In [14]:
# Check primal infeasibility — which group caps does the baseline solution violate?
def check_violations(
        result: dict,
        groups: dict[str, tuple[list[int], float]],
) -> pd.DataFrame:
    """
    Check which group cap constraints the baseline solution x* violates.

    This is the primal infeasibility check — the initialisation condition
    for the dual simplex method.

    violation_g = sum_{i in I_g} x*_i - U_g
    violation_g > 0 -> constraint is violated -> primal infeasible
    violation_g <= 0 -> constraint is satisfied -> primal feasible for this group

    Returns:
    DataFrame with columns:
        group, cap_g, usage_g, violation_g, primal_infeasible
    """
    x    = result["primal_x"]
    rows = []

    for group, (indices, cap) in groups.items():
        usage     = round(sum(x[i] for i in indices), 4)
        violation = round(usage - cap, 4)
        rows.append({
            "group"            : group,
            "cap_g"            : cap,
            "usage_g"          : usage,
            "violation_g"      : violation,
            "primal_infeasible": violation > 1e-4,
        })

    return pd.DataFrame(rows)

In [15]:
# Print violation check for all groups
print("Primal Infeasibility Check — Baseline Solution vs Group Caps")
print("Interpretation: positive violation = dual simplex initialisation condition met")
print()

for result in baseline_results:
    if result["status"] != "Optimal":
        continue

    viol_df = check_violations(result, groups)

    print(f"GROUP : {result['group_id'].upper().replace('_', ' ')}")

    display_rows = []
    for _, row in viol_df.iterrows():
        display_rows.append([
            row["group"],
            f"{row['cap_g']:.0f} g",
            f"{row['usage_g']:.2f} g",
            f"{row['violation_g']:+.2f} g",
            "YES — dual simplex applies" if row["primal_infeasible"] else "no",
        ])

    print(tabulate(
        display_rows,
        headers=["Group", "Cap U_g", "Baseline usage", "Violation", "Primal infeasible?"],
        tablefmt="rounded_outline",
    ))
    print()

Primal Infeasibility Check — Baseline Solution vs Group Caps
Interpretation: positive violation = dual simplex initialisation condition met

GROUP : CHILDREN 4 8 MF
╭────────────────┬───────────┬──────────────────┬─────────────┬────────────────────────────╮
│ Group          │ Cap U_g   │ Baseline usage   │ Violation   │ Primal infeasible?         │
├────────────────┼───────────┼──────────────────┼─────────────┼────────────────────────────┤
│ fluid_milk     │ 150 g     │ 198.87 g         │ +48.87 g    │ YES — dual simplex applies │
│ cooking_oils   │ 3 g       │ 12.67 g          │ +9.67 g     │ YES — dual simplex applies │
│ grain_products │ 120 g     │ 69.41 g          │ -50.59 g    │ no                         │
│ leafy_veg      │ 80 g      │ 2.34 g           │ -77.66 g    │ no                         │
│ protein_foods  │ 100 g     │ 13.14 g          │ -86.86 g    │ no                         │
│ fruits         │ 120 g     │ 56.00 g          │ -64.00 g    │ no                         

In [16]:
# Solve CAPPED LP (with group caps) — dual simplex re-optimisation
def solve_capped(
        baseline:        dict,
        c:               np.ndarray,
        A:               np.ndarray,
        food_names:      list[str],
        groups:          dict[str, tuple[list[int], float]],
) -> dict:
    """
    Add group cap constraints to the baseline system and re-solve.

    The group cap constraints:
        sum_{i in I_g} x_i <= U_g   for all g in G

    are appended to the baseline A_ub, b_ub.
    HiGHS internally applies the dual simplex because:
      - The baseline basis is dual feasible (Row 0 >= 0)
      - The baseline solution violates the new group caps (primal infeasible)
    This is the exact initialisation condition from the lecture.

    Returns:
    dict with group_id, status, cost_usd, primal_x, nit, A_ub, b_ub
    """
    group_id = baseline["group_id"]
    n        = len(food_names)

    # Start from the baseline nutrient constraint system
    A_ub_rows = list(baseline["A_ub"])
    b_ub_rows = list(baseline["b_ub"])
    con_names = list(baseline["con_names"])

    # Append group cap constraints: sum_{i in I_g} x_i <= U_g
    for group_name, (group_indices, cap) in groups.items():
        if not group_indices:
            continue
        row = np.zeros(n)
        for i in group_indices:
            row[i] = 1.0
        A_ub_rows.append(row)
        b_ub_rows.append(cap)
        con_names.append(("group_cap", group_name, cap))

    A_ub = np.array(A_ub_rows)
    b_ub = np.array(b_ub_rows)
    bounds = [(0.0, MAX_GRAMS_PER_FOOD)] * n

    res: OptimizeResult = linprog(
        c      = c,
        A_ub   = A_ub,
        b_ub   = b_ub,
        bounds = bounds,
        method = "highs",
    )

    if res.status != 0:
        return {"group_id": group_id, "status": res.message}

    return {
        "group_id"  : group_id,
        "status"    : "Optimal",
        "cost_usd"  : round(float(res.fun), 6),
        "primal_x"  : res.x,
        "nit"       : res.nit,
        "A_ub"      : A_ub,
        "b_ub"      : b_ub,
        "con_names" : con_names,
    }

In [17]:
# Solve capped LP for all groups
capped_results = []

for baseline in baseline_results:
    if baseline["status"] != "Optimal":
        capped_results.append({"group_id": baseline["group_id"], "status": "skipped"})
        continue

    result = solve_capped(
        baseline   = baseline,
        c          = c,
        A          = A,
        food_names = food_names,
        groups     = groups,
    )
    capped_results.append(result)
    print(f"{result['group_id']:25s}  Z*_capped = ${result['cost_usd']:.6f}  nit = {result['nit']}")

children_4_8_MF            Z*_capped = $0.738656  nit = 28
males_9_13                 Z*_capped = $0.903599  nit = 29
females_9_13               Z*_capped = $0.900184  nit = 25
males_14_18                Z*_capped = $1.036012  nit = 27
females_14_18              Z*_capped = $1.019802  nit = 31


In [18]:
# Print capped optimal menus
def print_capped_menu(result: dict, c: np.ndarray, food_names: list[str]) -> None:
    """
    Print the optimal lunch menu after dual simplex re-optimisation.

    Interpretation: these are the foods the LP selects once
    food group caps enforce meal realism. Comparison with the
    baseline menu shows which foods replaced the exploited items.
    """
    if result["status"] != "Optimal":
        return

    x     = result["primal_x"]
    cost  = result["cost_usd"]
    mask  = x > 0.1

    food_df = pd.DataFrame({
        "food_name" : [food_names[i] for i in range(len(x)) if mask[i]],
        "grams"     : x[mask].round(2),
        "cost_usd"  : (x[mask] * c[mask]).round(6),
        "cost_pct"  : (x[mask] * c[mask] / cost * 100).round(2),
    }).sort_values("cost_usd", ascending=False).reset_index(drop=True)
    food_df.index = food_df.index + 1

    print(f"\nGROUP : {result['group_id'].upper().replace('_', ' ')}")
    print(f"    Minimum cost after dual simplex : ${result['cost_usd']:.4f} USD")
    print(f"    Simplex iterations              : {result['nit']}")
    print()

    food_df["grams"]    = food_df["grams"].map(lambda v: f"{v:.1f} g")
    food_df["cost_usd"] = food_df["cost_usd"].map(lambda v: f"${v:.5f}")
    food_df["cost_pct"] = food_df["cost_pct"].map(lambda v: f"{v:.1f}%")

    print(tabulate(
        food_df,
        headers=["#", "Food", "Portion", "Cost (USD)", "Cost share"],
        tablefmt="rounded_outline",
        showindex=True,
    ))

In [19]:
for result in capped_results:
    print_capped_menu(result, c, food_names)


GROUP : CHILDREN 4 8 MF
    Minimum cost after dual simplex : $0.7387 USD
    Simplex iterations              : 28

╭─────┬────────────────────────────────────────┬───────────┬──────────────┬──────────────╮
│   # │ Food                                   │ Portion   │ Cost (USD)   │ Cost share   │
├─────┼────────────────────────────────────────┼───────────┼──────────────┼──────────────┤
│   1 │ Salmon, Pink, Canned (drained)         │ 22.8 g    │ $0.19127     │ 25.9%        │
│   2 │ Milk, 2% Reduced Fat, Fluid            │ 142.9 g   │ $0.13590     │ 18.4%        │
│   3 │ Whole Chicken (fresh)                  │ 77.2 g    │ $0.11687     │ 15.8%        │
│   4 │ Flaxseeds, Whole                       │ 14.1 g    │ $0.10899     │ 14.8%        │
│   5 │ Cornmeal, Whole Grain, Yellow (dry)    │ 35.3 g    │ $0.10369     │ 14.0%        │
│   6 │ Raisin Bran Cereal                     │ 4.3 g     │ $0.02813     │ 3.8%         │
│   7 │ Collard Greens, Raw                    │ 3.8 g     │ $0.

In [20]:
# Compare baseline vs capped — cost of acceptability
def print_dual_simplex_comparison(
        baseline_results: list[dict],
        capped_results:   list[dict],
) -> None:
    """
    Side-by-side comparison of baseline and capped optimal solutions.

    Cost of acceptability = Z*_capped - Z*_baseline

    This is the additional cost imposed by requiring a realistic meal
    structure via the food group caps. It represents the trade-off
    between pure cost optimality and meal acceptability.

    Iteration comparison shows that the capped solution requires
    additional dual simplex pivots to restore primal feasibility
    from the dual-feasible baseline basis.
    """
    print("Dual Simplex Re-optimisation — Baseline vs Capped Solution")
    print("Interpretation: delta cost = price of meal acceptability")
    print()

    rows = []
    for base, capped in zip(baseline_results, capped_results):
        if base["status"] != "Optimal" or capped["status"] != "Optimal":
            continue

        delta     = round(capped["cost_usd"] - base["cost_usd"], 6)
        delta_pct = round(delta / base["cost_usd"] * 100, 2)
        extra_nit = capped["nit"] - base["nit"]

        rows.append([
            base["group_id"],
            f"${base['cost_usd']:.4f}",
            base["nit"],
            f"${capped['cost_usd']:.4f}",
            capped["nit"],
            f"+${delta:.4f}",
            f"+{delta_pct:.1f}%",
            f"+{extra_nit}",
        ])

    print(tabulate(
        rows,
        headers=[
            "Group",
            "Z* baseline", "nit baseline",
            "Z* capped",   "nit capped",
            "Delta cost",  "Delta %",
            "Extra pivots",
        ],
        tablefmt="rounded_outline",
    ))

    print()
    print("Extra pivots = additional dual simplex iterations to restore primal feasibility")
    print("Both violated groups (fluid_milk, cooking_oils) required pivoting out of the")
    print("baseline basis — one pivot per violated constraint at minimum.")

In [21]:
print_dual_simplex_comparison(baseline_results, capped_results)

Dual Simplex Re-optimisation — Baseline vs Capped Solution
Interpretation: delta cost = price of meal acceptability

╭─────────────────┬───────────────┬────────────────┬─────────────┬──────────────┬──────────────┬───────────┬────────────────╮
│ Group           │ Z* baseline   │   nit baseline │ Z* capped   │   nit capped │ Delta cost   │ Delta %   │   Extra pivots │
├─────────────────┼───────────────┼────────────────┼─────────────┼──────────────┼──────────────┼───────────┼────────────────┤
│ children_4_8_MF │ $0.6872       │             19 │ $0.7387     │           28 │ +$0.0514     │ +7.5%     │             +9 │
│ males_9_13      │ $0.7962       │             19 │ $0.9036     │           29 │ +$0.1074     │ +13.5%    │            +10 │
│ females_9_13    │ $0.7734       │             14 │ $0.9002     │           25 │ +$0.1268     │ +16.4%    │            +11 │
│ males_14_18     │ $0.9275       │             19 │ $1.0360     │           27 │ +$0.1085     │ +11.7%    │             +8 │
│

In [22]:
# Verify group caps are satisfied in capped solution
def verify_group_caps(result: dict, groups: dict) -> None:
    """
    Verify that all group cap constraints are satisfied in the capped solution.

    After dual simplex re-optimisation, the solution must be both:
    - Primal feasible: all group caps satisfied
    - Dual feasible: maintained throughout (by construction of dual simplex)

    This confirms dual simplex has successfully restored primal feasibility.
    """
    if result["status"] != "Optimal":
        return

    x = result["primal_x"]

    print(f"GROUP : {result['group_id'].upper().replace('_', ' ')}")

    rows = []
    all_feasible = True
    for group, (indices, cap) in groups.items():
        usage   = round(sum(x[i] for i in indices), 4)
        slack   = round(cap - usage, 4)
        feasible = slack >= -1e-4
        if not feasible:
            all_feasible = False
        rows.append([
            group,
            f"{cap:.0f} g",
            f"{usage:.2f} g",
            f"{slack:.2f} g",
            "OK" if feasible else "VIOLATED",
        ])

    print(tabulate(
        rows,
        headers=["Group", "Cap U_g", "Usage", "Slack", "Primal feasible"],
        tablefmt="rounded_outline",
    ))
    print(f"    All caps satisfied: {'YES — primal feasibility restored' if all_feasible else 'NO'}")
    print()

In [23]:
print("Group Cap Feasibility Verification — After Dual Simplex Re-optimisation")
print()
for result in capped_results:
    verify_group_caps(result, groups)

Group Cap Feasibility Verification — After Dual Simplex Re-optimisation

GROUP : CHILDREN 4 8 MF
╭────────────────┬───────────┬──────────┬─────────┬───────────────────╮
│ Group          │ Cap U_g   │ Usage    │ Slack   │ Primal feasible   │
├────────────────┼───────────┼──────────┼─────────┼───────────────────┤
│ fluid_milk     │ 150 g     │ 150.00 g │ 0.00 g  │ OK                │
│ cooking_oils   │ 3 g       │ 3.00 g   │ 0.00 g  │ OK                │
│ grain_products │ 120 g     │ 39.59 g  │ 80.41 g │ OK                │
│ leafy_veg      │ 80 g      │ 3.81 g   │ 76.19 g │ OK                │
│ protein_foods  │ 100 g     │ 100.00 g │ 0.00 g  │ OK                │
│ fruits         │ 120 g     │ 102.71 g │ 17.29 g │ OK                │
╰────────────────┴───────────┴──────────┴─────────┴───────────────────╯
    All caps satisfied: YES — primal feasibility restored

GROUP : MALES 9 13
╭────────────────┬───────────┬──────────┬─────────┬───────────────────╮
│ Group          │ Cap U_g   │ Us

In [24]:
# Save dual simplex results to markdown
def save_dual_simplex_md(
        baseline: dict,
        capped:   dict,
        groups:   dict,
        c:        np.ndarray,
        food_names: list[str],
        save_path: str = ".",
) -> None:
    """
    Save dual simplex analysis for one group as a Markdown file.

    Output: dual_simplex_{group_id}.md
    Sections: header, violation check, capped menu, cost of acceptability,
              group cap feasibility verification
    """
    if baseline["status"] != "Optimal" or capped["status"] != "Optimal":
        print(f"  Skipping {baseline['group_id']} — no optimal solution.")
        return

    gid      = baseline["group_id"]
    filename = f"dual_simplex_{gid}.md"

    delta     = round(capped["cost_usd"] - baseline["cost_usd"], 6)
    delta_pct = round(delta / baseline["cost_usd"] * 100, 2)

    lines = []

    # Header
    lines.append(f"# Dual Simplex Analysis — {gid.replace('_', ' ').title()}\n")
    lines.append(f"**Baseline cost (no group caps):** ${baseline['cost_usd']:.6f} USD  ")
    lines.append(f"**Capped cost (after dual simplex):** ${capped['cost_usd']:.6f} USD  ")
    lines.append(f"**Cost of acceptability:** +${delta:.6f} USD ({delta_pct:+.1f}%)  ")
    lines.append(f"**Baseline iterations:** {baseline['nit']}  ")
    lines.append(f"**Capped iterations:** {capped['nit']}  ")
    lines.append(f"**Extra pivots (dual simplex):** +{capped['nit'] - baseline['nit']}\n")

    # Violation check
    x_base = baseline["primal_x"]
    lines.append("## Primal Infeasibility Check (Baseline vs Group Caps)\n")
    lines.append("Positive violation = dual simplex initialisation condition met.\n")
    lines.append("| Group | Cap U_g | Baseline usage | Violation | Primal infeasible |")
    lines.append("|-------|--------:|---------------:|----------:|:-----------------:|")
    for group, (indices, cap) in groups.items():
        usage = round(sum(x_base[i] for i in indices), 2)
        viol  = round(usage - cap, 2)
        inf   = viol > 1e-4
        lines.append(
            f"| {group} | {cap:.0f} g | {usage:.2f} g "
            f"| {viol:+.2f} g | {'YES' if inf else 'no'} |"
        )
    lines.append("")

    # Capped menu
    x_cap = capped["primal_x"]
    mask  = x_cap > 0.1
    total_cost = capped["cost_usd"]
    lines.append("## Optimal Lunch Menu After Dual Simplex Re-optimisation\n")
    lines.append("| # | Food | Portion (g) | Cost (USD) | Cost share (%) |")
    lines.append("|---|------|------------:|------------:|---------------:|")
    food_rows = sorted(
        [(food_names[i], x_cap[i], x_cap[i]*c[i])
         for i in range(len(x_cap)) if mask[i]],
        key=lambda r: -r[2]
    )
    for idx, (fname, grams, fcost) in enumerate(food_rows, 1):
        lines.append(
            f"| {idx} | {fname} | {grams:.1f} "
            f"| ${fcost:.5f} | {fcost/total_cost*100:.1f}% |"
        )
    lines.append("")

    # Cap verification
    lines.append("## Group Cap Feasibility After Dual Simplex\n")
    lines.append("| Group | Cap U_g | Usage | Slack | Primal feasible |")
    lines.append("|-------|--------:|------:|------:|:---------------:|")
    for group, (indices, cap) in groups.items():
        usage  = round(sum(x_cap[i] for i in indices), 2)
        slack  = round(cap - usage, 2)
        lines.append(
            f"| {group} | {cap:.0f} g | {usage:.2f} g "
            f"| {slack:.2f} g | {'OK' if slack >= -1e-4 else 'VIOLATED'} |"
        )
    lines.append("")

    # Write and save
    os.makedirs(save_path, exist_ok=True)
    filepath = os.path.join(save_path, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"  Saved: {filename}")

In [25]:
SAVE_PATH = "/Users/julianyoo/Downloads/Study/PWR/Term 1/Operations Research/Project/result/dual_simplex_01"
for base, capped in zip(baseline_results, capped_results):
    save_dual_simplex_md(
        baseline   = base,
        capped     = capped,
        groups     = groups,
        c          = c,
        food_names = food_names,
        save_path  = SAVE_PATH,
    )

  Saved: dual_simplex_children_4_8_MF.md
  Saved: dual_simplex_males_9_13.md
  Saved: dual_simplex_females_9_13.md
  Saved: dual_simplex_males_14_18.md
  Saved: dual_simplex_females_14_18.md


## References

### Dual Simplex Method

Hillier, F. S., & Lieberman, G. J. (2015).
*Introduction to operations research* (10th ed.). McGraw-Hill Education.

Vanderbei, R. J. (2020).
*Linear programming: Foundations and extensions* (5th ed.). Springer.
https://doi.org/10.1007/978-3-030-39415-8

### LP Solver

Huangfu, Q., & Hall, J. A. J. (2018).
Parallelizing the dual revised simplex method.
*Mathematical Programming Computation*, 10(1), 119–142.
https://doi.org/10.1007/s12532-017-0130-5

Virtanen, P., Gommers, R., Oliphant, T. E., Haberland, M., Reddy, T.,
Cournapeau, D., … van der Walt, S. J. (2020).
SciPy 1.0: Fundamental algorithms for scientific computing in Python.
*Nature Methods*, 17, 261–272.
https://doi.org/10.1038/s41592-019-0686-2

### LP Diet Optimisation

Darmon, N., Ferguson, E. L., & Briend, A. (2002).
A cost constraint alone has adverse effects on food selection and
nutrient density: An analysis of human diets by linear programming.
*Journal of Nutrition*, 132(12), 3764–3771.
https://doi.org/10.1093/jn/132.12.3764

### Nutritional Data

U.S. Department of Agriculture, Agricultural Research Service. (2019).
*FoodData Central* (SR Legacy Foods / Foundation Foods).
Retrieved from https://fdc.nal.usda.gov

### Food Price Data

U.S. Bureau of Labor Statistics. (2024).
*Average retail food prices — CPI average price series*.
Retrieved from https://data.bls.gov

### School Nutrition Policy

U.S. Department of Agriculture, Food and Nutrition Service. (2024).
Child nutrition programs: Meal patterns consistent with the 2020–2025
dietary guidelines for Americans. *Federal Register*, 89(81).
Retrieved from https://www.federalregister.gov/documents/2024/04/25/2024-08098